# Loss Function Comparison

Compare metric learning loss functions while holding dataset, backbone,
embedding dimension, and preprocessing constant.

> **Warning:** Results are diagnostic only. LeaData is a public leather texture
> dataset, not a luxury authentication dataset. Loss function performance here
> does not prove luxury authentication accuracy.

See [Notebook Execution Plan](EXECUTION_PLAN.md) for run order and status.

## Run Experiments

From `auth-platform/`:

```bash
# Quick comparison (2 epochs, 100 images per split)
make loss-comparison-leadata-quick

# Full comparison (10 epochs, full dataset)
make loss-comparison-leadata

# Compare all learned results against baseline
make compare-loss-comparison
```

Losses compared:
- `contrastive` — pairwise contrastive (baseline learned loss)
- `supervised_contrastive` — SupCon batch loss
- `batch_hard_triplet` — batch-hard triplet mining

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

LOSS_DIR = Path("../experiments/outputs/loss-comparison")
BASELINE_DIR = Path("../experiments/outputs/baseline-feature-comparison")
DATASET = "external_leadata_v0.1.0"
ds_tag = DATASET.replace(".", "_")

# Find loss comparison summary
summary_path = LOSS_DIR / "loss_comparison_summary.json"
print(f"Looking for: {summary_path}")
print(f"Exists: {summary_path.exists()}")

In [ ]:
# Load comparison summary
if not summary_path.exists():
    print("No loss comparison results found. Run experiment first:")
    print("  make loss-comparison-leadata-quick")
    data = None
else:
    data = json.loads(summary_path.read_text())
    results = [r for r in data.get("results", []) if r.get("evaluation_metrics")]
    print(f"Configs run: {data.get('configs_run', 0)}")
    print(f"Results with metrics: {len(results)}")
    print(f"Failures: {len(data.get('failures', []))}")
    print()

    if results:
        header = f"{'Loss':<25} {'Dim':>5} {'ClustSep':>10} {'NN Agree':>10} {'R@1':>8} {'R@3':>8}"
        print(header)
        print("-" * len(header))
        for r in results:
            em = r["evaluation_metrics"]
            print(f"{r.get('loss', '?'):<25} {em.get('dimension', ''):>5} "
                  f"{em.get('cluster_separation', 0):>10.4f} "
                  f"{em.get('nn_agreement_rate', 0):>9.1%} "
                  f"{em.get('recall_at_1', 0):>8.4f} "
                  f"{em.get('recall_at_3', 0):>8.4f}")

In [ ]:
# Bar chart: cluster separation by loss
if data and results:
    losses = [r.get("loss", "?") for r in results]
    cluster_seps = [r["evaluation_metrics"].get("cluster_separation", 0) for r in results]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(losses, cluster_seps, color="steelblue")
    ax.set_xlabel("Cluster Separation")
    ax.set_title(f"Cluster Separation by Loss Function — {DATASET}")
    for bar, val in zip(bars, cluster_seps):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# Bar chart: recall@1 by loss
if data and results:
    recalls = [r["evaluation_metrics"].get("recall_at_1", 0) for r in results]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(losses, recalls, color="darkorange")
    ax.set_xlabel("Recall@1")
    ax.set_title(f"Recall@1 by Loss Function — {DATASET}")
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, recalls):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# Training time comparison
if data and results:
    times = [r.get("training_time_seconds", 0) for r in results]
    if any(t > 0 for t in times):
        fig, ax = plt.subplots(figsize=(10, 4))
        bars = ax.barh(losses, times, color="seagreen")
        ax.set_xlabel("Training Time (s)")
        ax.set_title(f"Training Time by Loss Function — {DATASET}")
        for bar, val in zip(bars, times):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{val:.0f}s", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

## Notes / Conclusions

- All losses use pretrained ResNet18, 256-dim embeddings, same augmentations.
- Contrastive loss is the Phase 5 baseline learned loss.
- Supervised contrastive (SupCon) uses all same-class samples as positives within a batch.
- Batch-hard triplet selects hardest positive/negative per anchor in each batch.
- Compare cluster separation and NN agreement to identify the best loss for this data.
- `texture_baseline_v0.2.0` (hand-crafted) remains the overall comparison target.

**These results are diagnostic only.** They do not prove authentication accuracy.